# ResPlan — Conditional Variational Autoencoder for Residential Floor-Plan Generation

**Day 1 submission | Team 4**

## Research objective
Train a Conditional Variational Autoencoder (CVAE) on the ResPlan residential floor-plan dataset. The model learns a compact latent distribution and decodes sampled latent vectors into new multi-layer floor-plan representations.

**Primary demonstration:** latent-space interpolation between two real floor plans (the VAE-blend use case), with a qualitative interpolation figure.

## Dataset representation
The notebook rasterises 14 polygon geometry layers into a multi-channel image:

`wall, door, window, front_door, bedroom, bathroom, kitchen, living, inner, balcony, garden, parking, pool, land`

The `neighbor` field is retained as graph metadata for future work; it is not rasterised because it is a tuple rather than a polygon geometry. Three numeric attributes (`net_area`, `area`, `wall_depth`) condition the VAE.

**Important:** the generated output is a raster concept representation, not an editable CAD/BIM floor plan and not a guarantee of code-compliant architecture.

## 1. Reproducibility, paths and imports

Run this notebook from `01_autoencoder/notebook/`. Outputs are written into sibling folders under `01_autoencoder/`.

The source pickle is external to the Git repository. Update `DATA_PATH` if your local dataset is stored elsewhere. Do not commit the pickle or generated model binaries to GitHub.

In [1]:
from pathlib import Path

# Project root: edit only if your folder is stored elsewhere.
PROJECT_DIR = Path(
    r"D:\02_LEARNING\02_Masters_IAAC_AI_Architecture\10-Term_01\01-Ai Methods\hackathon-team4-resplan\01_autoencoder"
)

CACHE_DIR = PROJECT_DIR / "cache"
FIGURES_DIR = PROJECT_DIR / "figures"
MODEL_DIR = PROJECT_DIR / "models"
MODEL_APP_DIR = PROJECT_DIR / "model_app"

for folder in (CACHE_DIR, FIGURES_DIR, MODEL_DIR, MODEL_APP_DIR):
    folder.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_DIR)
print("Project folder exists:", PROJECT_DIR.exists())
print("Cache:", CACHE_DIR)
print("Figures:", FIGURES_DIR)
print("Models:", MODEL_DIR)
print("Model app:", MODEL_APP_DIR)


Project: D:\02_LEARNING\02_Masters_IAAC_AI_Architecture\10-Term_01\01-Ai Methods\hackathon-team4-resplan\01_autoencoder
Project folder exists: True
Cache: D:\02_LEARNING\02_Masters_IAAC_AI_Architecture\10-Term_01\01-Ai Methods\hackathon-team4-resplan\01_autoencoder\cache
Figures: D:\02_LEARNING\02_Masters_IAAC_AI_Architecture\10-Term_01\01-Ai Methods\hackathon-team4-resplan\01_autoencoder\figures
Models: D:\02_LEARNING\02_Masters_IAAC_AI_Architecture\10-Term_01\01-Ai Methods\hackathon-team4-resplan\01_autoencoder\models
Model app: D:\02_LEARNING\02_Masters_IAAC_AI_Architecture\10-Term_01\01-Ai Methods\hackathon-team4-resplan\01_autoencoder\model_app


In [2]:
import os
import gc
import json
import time
import random
import pickle
import platform
import warnings

import numpy as np
import pandas as pd
import psutil
from PIL import Image, ImageDraw, ImageFont

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Deterministic behaviour where supported; benchmark is disabled for repeatability.
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Dataset is external to the Git repository. Update only this path if needed.
DATA_PATH = Path(
    r"D:\02_LEARNING\02_Masters_IAAC_AI_Architecture\10-Term_01\01-AI Methods\ResPlan\ResPlan\ResPlan.pkl"
)

IMAGE_SIZE = 64
BATCH_SIZE = 16
EPOCHS = 30
PATIENCE = 6
NUM_WORKERS = 0
PIN_MEMORY = False

def check_memory(stage="Memory status"):
    process = psutil.Process(os.getpid())
    system = psutil.virtual_memory()
    print(f"\\n========== {stage} ==========")
    print(f"System RAM total:     {system.total / 1024**3:.2f} GB")
    print(f"System RAM available: {system.available / 1024**3:.2f} GB")
    print(f"System RAM used:      {system.percent:.1f}%")
    print(f"Jupyter kernel RAM:   {process.memory_info().rss / 1024**3:.2f} GB")
    if torch.cuda.is_available():
        print(f"GPU:                  {torch.cuda.get_device_name(0)}")
        print(f"GPU allocated:        {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
        print(f"GPU reserved:         {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)
print("Device:", DEVICE)
print("Dataset exists:", DATA_PATH.exists())
print("Dataset path:", DATA_PATH)
print("Batch size:", BATCH_SIZE)
check_memory("After setup")


Python: 3.12.15
PyTorch: 2.11.0+cu128
NumPy: 2.5.5
Device: cuda
Dataset exists: True
Dataset path: D:\02_LEARNING\02_Masters_IAAC_AI_Architecture\10-Term_01\01-AI Methods\ResPlan\ResPlan\ResPlan.pkl
Batch size: 16
\n========== After setup ==========
System RAM total:     15.69 GB
System RAM available: 4.35 GB
System RAM used:      72.3%
Jupyter kernel RAM:   0.61 GB
GPU:                  NVIDIA GeForce RTX 3060 Laptop GPU
GPU allocated:        0.00 GB
GPU reserved:         0.00 GB


## 2. Load the dataset and split it before preprocessing

The split is created immediately after loading the records. No image preprocessing, scaler fitting or model training occurs before this split.

We use a fixed 70% / 15% / 15% train-validation-test split. All model configurations reuse these exact indices.

In [3]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH}\n"
        "Edit DATA_PATH in Section 1 to point to ResPlan.pkl."
    )

print(f"Dataset size: {DATA_PATH.stat().st_size / (1024**2):.2f} MB")
with DATA_PATH.open("rb") as f:
    records = pickle.load(f)

if not isinstance(records, list):
    raise TypeError(f"Expected a list of records, got {type(records)}")

n_records = len(records)
all_indices = np.arange(n_records)

# Split before fitting preprocessing or rasterising.
train_indices, temp_indices = train_test_split(
    all_indices, test_size=0.30, random_state=SEED, shuffle=True
)
val_indices, test_indices = train_test_split(
    temp_indices, test_size=0.50, random_state=SEED, shuffle=True
)

assert not (set(train_indices) & set(val_indices))
assert not (set(train_indices) & set(test_indices))
assert not (set(val_indices) & set(test_indices))

print("Records:", n_records)
print(f"Train:      {len(train_indices):,} ({len(train_indices)/n_records:.1%})")
print(f"Validation: {len(val_indices):,} ({len(val_indices)/n_records:.1%})")
print(f"Test:       {len(test_indices):,} ({len(test_indices)/n_records:.1%})")
check_memory("After loading records and splitting")


Dataset size: 246.48 MB
Records: 17000
Train:      11,900 (70.0%)
Validation: 2,550 (15.0%)
Test:       2,550 (15.0%)
\n========== After loading records and splitting ==========
System RAM total:     15.69 GB
System RAM available: 3.87 GB
System RAM used:      75.3%
Jupyter kernel RAM:   1.43 GB
GPU:                  NVIDIA GeForce RTX 3060 Laptop GPU
GPU allocated:        0.00 GB
GPU reserved:         0.00 GB


## 3. Inspect the dataset schema

This inspection documents the actual record structure. Geometry fields are handled separately from numeric attributes and graph metadata.

In [4]:
sample = records[int(train_indices[0])]

schema_rows = []
for key, value in sample.items():
    schema_rows.append({
        "feature": key,
        "python_type": type(value).__name__,
        "geometry_type": getattr(value, "geom_type", None),
        "is_empty": getattr(value, "is_empty", None)
    })

schema_df = pd.DataFrame(schema_rows)
display(schema_df)

print("\nNumber of records:", len(records))
print("Sample record ID:", sample.get("id"))
print("Sample area:", sample.get("area"))
print("Sample net area:", sample.get("net_area"))
print("Sample wall depth:", sample.get("wall_depth"))

,feature,python_type,geometry_type,is_empty
0,balcony,MultiPolygon,MultiPolygon,False
1,bathroom,MultiPolygon,MultiPolygon,False
2,bedroom,MultiPolygon,MultiPolygon,False
3,door,MultiPolygon,MultiPolygon,False
4,garden,MultiPolygon,MultiPolygon,True
5,inner,MultiPolygon,MultiPolygon,False
6,parking,MultiPolygon,MultiPolygon,True
7,pool,MultiPolygon,MultiPolygon,True
8,wall,MultiPolygon,MultiPolygon,False
9,window,MultiPolygon,MultiPolygon,False



Number of records: 17000
Sample record ID: 3822
Sample area: 185.341485
Sample net area: 18.220349359735998
Sample wall depth: 3.4324022346368714


## 4. Define the spatial and numeric features

We use all polygon geometry layers found in the inspected schema. `neighbor` is not included in the raster tensor because it is a tuple/graph-like field. `id` is only an identifier and must not be learned as a design feature.

The numeric conditioning vector contains `net_area`, `area`, and `wall_depth`. Its scaler is fitted using training records only.

In [5]:
GEOMETRY_CHANNELS = [
    "wall",
    "door",
    "window",
    "front_door",
    "bedroom",
    "bathroom",
    "kitchen",
    "living",
    "inner",
    "balcony",
    "garden",
    "parking",
    "pool",
    "land",
]

NUMERIC_FEATURES = ["net_area", "area", "wall_depth"]
N_CHANNELS = len(GEOMETRY_CHANNELS)

print("Raster channels:", N_CHANNELS)
for i, name in enumerate(GEOMETRY_CHANNELS):
    print(f"{i:02d}: {name}")
print("Numeric conditioning:", NUMERIC_FEATURES)

Raster channels: 14
00: wall
01: door
02: window
03: front_door
04: bedroom
05: bathroom
06: kitchen
07: living
08: inner
09: balcony
10: garden
11: parking
12: pool
13: land
Numeric conditioning: ['net_area', 'area', 'wall_depth']


## 5. Fit numeric conditioning scaler on training records only

Missing numeric values are imputed using training-set medians. The scaler is fitted only on training records to avoid information leakage.

In [6]:
def numeric_row(record):
    values = []
    for feature in NUMERIC_FEATURES:
        value = record.get(feature, np.nan)
        try:
            value = float(value)
        except (TypeError, ValueError):
            value = np.nan
        if not np.isfinite(value):
            value = np.nan
        values.append(value)
    return values

numeric_all = np.asarray([numeric_row(r) for r in records], dtype=np.float32)

train_numeric = numeric_all[train_indices]
numeric_medians = np.nanmedian(train_numeric, axis=0)
numeric_medians = np.where(np.isfinite(numeric_medians), numeric_medians, 0.0)

numeric_all_imputed = np.where(
    np.isfinite(numeric_all), numeric_all, numeric_medians
).astype(np.float32)

numeric_scaler = StandardScaler()
numeric_scaler.fit(numeric_all_imputed[train_indices])

numeric_scaled = numeric_scaler.transform(numeric_all_imputed).astype(np.float32)

print("Numeric medians:", dict(zip(NUMERIC_FEATURES, numeric_medians.tolist())))
print("Scaled numeric shape:", numeric_scaled.shape)
print("Training scaled means:", numeric_scaled[train_indices].mean(axis=0))

Numeric medians: {'net_area': 67.87861633300781, 'area': 109.16102600097656, 'wall_depth': 4.13167667388916}
Scaled numeric shape: (17000, 3)
Training scaled means: [-2.2065928e-08 -6.9653538e-08 -3.3981661e-07]


## 6. Geometry rasterisation

Each record is rasterised into a `14 × 64 × 64` binary tensor.

For each floor plan, the transform uses the `land` bounds when available; otherwise it uses the combined bounds of available geometry layers. Aspect ratio is preserved and the plan is centred on a square canvas.

Both `Polygon` and `MultiPolygon` are supported. Polygon holes are drawn as background. Invalid or missing geometry is skipped safely.

The raster cache is stored as a memory-mapped `uint8` array to avoid holding the entire image dataset in RAM. At 64×64 with 14 channels, the cache is approximately 0.9 GB for 17,000 plans.

In [7]:
from shapely.geometry import Polygon, MultiPolygon, GeometryCollection
from shapely.ops import unary_union

CACHE_PATH = CACHE_DIR / f"resplan_{IMAGE_SIZE}px_{N_CHANNELS}ch_uint8.dat"
CACHE_META_PATH = CACHE_DIR / f"resplan_{IMAGE_SIZE}px_{N_CHANNELS}ch_meta.json"

def iter_polygons(geom):
    """Yield Polygon objects from Polygon, MultiPolygon or GeometryCollection."""
    if geom is None:
        return
    if getattr(geom, "is_empty", True):
        return
    if isinstance(geom, Polygon):
        yield geom
    elif isinstance(geom, (MultiPolygon, GeometryCollection)):
        for part in geom.geoms:
            yield from iter_polygons(part)
    elif hasattr(geom, "geoms"):
        for part in geom.geoms:
            yield from iter_polygons(part)

def get_record_bounds(record):
    """Get common spatial bounds for all selected layers in one plan."""
    land_geom = record.get("land")
    if land_geom is not None and not getattr(land_geom, "is_empty", True):
        bounds = land_geom.bounds
        if len(bounds) == 4 and np.all(np.isfinite(bounds)):
            minx, miny, maxx, maxy = bounds
            if maxx > minx and maxy > miny:
                return bounds

    geoms = [
        record.get(name) for name in GEOMETRY_CHANNELS
        if record.get(name) is not None
        and not getattr(record.get(name), "is_empty", True)
    ]
    if not geoms:
        return (0.0, 0.0, 1.0, 1.0)

    try:
        bounds = unary_union(geoms).bounds
    except Exception:
        bounds = None

    if bounds is None or len(bounds) != 4 or not np.all(np.isfinite(bounds)):
        return (0.0, 0.0, 1.0, 1.0)

    minx, miny, maxx, maxy = bounds
    if maxx <= minx or maxy <= miny:
        return (0.0, 0.0, 1.0, 1.0)
    return bounds

def make_transform(bounds, image_size, padding=2):
    minx, miny, maxx, maxy = bounds
    width = max(maxx - minx, 1e-8)
    height = max(maxy - miny, 1e-8)
    usable = image_size - 1 - 2 * padding
    scale = min(usable / width, usable / height)

    drawn_w = width * scale
    drawn_h = height * scale
    offset_x = (image_size - drawn_w) / 2.0
    offset_y = (image_size - drawn_h) / 2.0

    def xy_to_pixel(x, y):
        px = offset_x + (x - minx) * scale
        # Image coordinates increase downwards; invert world Y.
        py = image_size - offset_y - (y - miny) * scale
        return (int(round(px)), int(round(py)))

    return xy_to_pixel

def draw_geometry(draw, geom, xy_to_pixel):
    for polygon in iter_polygons(geom):
        try:
            exterior = [xy_to_pixel(x, y) for x, y in polygon.exterior.coords]
            if len(exterior) >= 3:
                draw.polygon(exterior, fill=1)

            # Clear interior rings to preserve polygon holes.
            for ring in polygon.interiors:
                interior = [xy_to_pixel(x, y) for x, y in ring.coords]
                if len(interior) >= 3:
                    draw.polygon(interior, fill=0)
        except Exception:
            # Skip a malformed polygon component without losing other layers.
            continue

def rasterize_record(record, image_size=IMAGE_SIZE):
    bounds = get_record_bounds(record)
    xy_to_pixel = make_transform(bounds, image_size)

    tensor = np.zeros(
        (N_CHANNELS, image_size, image_size), dtype=np.uint8
    )

    for channel_index, feature in enumerate(GEOMETRY_CHANNELS):
        geom = record.get(feature)
        if geom is None or getattr(geom, "is_empty", True):
            continue

        image = Image.new("L", (image_size, image_size), 0)
        draw = ImageDraw.Draw(image)
        draw_geometry(draw, geom, xy_to_pixel)
        tensor[channel_index] = np.asarray(image, dtype=np.uint8)

    return tensor

# File-size validation alone is unsafe: an interrupted process can leave
# a preallocated file with the expected size but incomplete image contents.
CACHE_COMPLETE_PATH = CACHE_DIR / f"resplan_{IMAGE_SIZE}px_{N_CHANNELS}ch_complete.txt"
expected_shape = (len(records), N_CHANNELS, IMAGE_SIZE, IMAGE_SIZE)
expected_bytes = int(np.prod(expected_shape)) * np.dtype(np.uint8).itemsize

expected_cache_metadata = {
    "records": len(records),
    "channels": GEOMETRY_CHANNELS,
    "image_size": IMAGE_SIZE,
    "dtype": "uint8",
    "bounds_strategy": "land bounds when available; otherwise union bounds"
}

metadata_matches = False
if CACHE_META_PATH.exists():
    try:
        saved_metadata = json.loads(CACHE_META_PATH.read_text(encoding="utf-8"))
        metadata_matches = all(
            saved_metadata.get(key) == value
            for key, value in expected_cache_metadata.items()
        )
    except (OSError, json.JSONDecodeError):
        metadata_matches = False

cache_valid = (
    CACHE_PATH.exists()
    and CACHE_PATH.stat().st_size == expected_bytes
    and CACHE_COMPLETE_PATH.exists()
    and metadata_matches
)

if cache_valid:
    print("Using completed raster cache:", CACHE_PATH)
    image_cache = np.memmap(
        CACHE_PATH, dtype=np.uint8, mode="r", shape=expected_shape
    )
else:
    print("Raster cache missing or incomplete. Building cache.")
    if CACHE_PATH.exists():
        CACHE_PATH.unlink()
    if CACHE_COMPLETE_PATH.exists():
        CACHE_COMPLETE_PATH.unlink()

    image_cache = np.memmap(
        CACHE_PATH, dtype=np.uint8, mode="w+", shape=expected_shape
    )
    start = time.time()

    for i, record in enumerate(records):
        image_cache[i] = rasterize_record(record)

        if (i + 1) % 250 == 0:
            image_cache.flush()
            print(f"Rasterised {i+1:,}/{len(records):,}")

    image_cache.flush()
    del image_cache
    gc.collect()

    # Marker is written only after all raster data has been flushed.
    CACHE_COMPLETE_PATH.write_text(
        "Rasterisation completed successfully.", encoding="utf-8"
    )
    print(f"Rasterisation completed in {(time.time()-start)/60:.1f} minutes.")

    image_cache = np.memmap(
        CACHE_PATH, dtype=np.uint8, mode="r", shape=expected_shape
    )

print("Cache shape:", image_cache.shape)
print("Cache dtype:", image_cache.dtype)
print(f"Cache size: {CACHE_PATH.stat().st_size / (1024**3):.2f} GB")

with CACHE_META_PATH.open("w", encoding="utf-8") as f:
    json.dump(expected_cache_metadata, f, indent=2)

print("Cache shape:", image_cache.shape)
print("Cache size (GB):", round(CACHE_PATH.stat().st_size / (1024**3), 3))

Using completed raster cache: D:\02_LEARNING\02_Masters_IAAC_AI_Architecture\10-Term_01\01-Ai Methods\hackathon-team4-resplan\01_autoencoder\cache\resplan_64px_14ch_uint8.dat
Cache shape: (17000, 14, 64, 64)
Cache dtype: uint8
Cache size: 0.91 GB
Cache shape: (17000, 14, 64, 64)
Cache size (GB): 0.908


## 7. Inspect rasterised floor plans

The first figure shows a composite view of all channels. The second shows the individual semantic layers. Check that the geometry is aligned and recognisable before training.

In [8]:
# ============================================================
# SECTION 7 — MEMORY-SAFE RESPLAN VISUALISATION
# Uses Pillow instead of Matplotlib
# ============================================================

import os
import gc
import psutil
import numpy as np

from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display, Image as NotebookImage

# ------------------------------------------------------------
# 1. CHECK REQUIRED VARIABLES
# ------------------------------------------------------------

required_variables = [
    "train_indices",
    "image_cache",
    "records",
    "N_CHANNELS",
    "GEOMETRY_CHANNELS",
    "FIGURES_DIR"
]

missing = [
    variable for variable in required_variables
    if variable not in globals()
]

if missing:
    raise RuntimeError(
        f"Missing variables: {missing}. "
        "Run Sections 1–6 first."
    )

# ------------------------------------------------------------
# 2. MEMORY DIAGNOSTIC
# ------------------------------------------------------------

def memory_status(stage):

    process = psutil.Process(os.getpid())
    system = psutil.virtual_memory()

    print(f"\n========== {stage} ==========")
    print(f"Available RAM: {system.available / 1024**3:.2f} GB")
    print(f"System RAM usage: {system.percent:.1f}%")
    print(f"Jupyter RAM: {process.memory_info().rss / 1024**3:.2f} GB")


memory_status("Before visualisation")

# Stop if the system is critically low on available memory.
available_gb = psutil.virtual_memory().available / 1024**3

if available_gb < 1.0:
    raise MemoryError(
        f"Only {available_gb:.2f} GB RAM available. "
        "Close other applications before continuing."
    )

# ------------------------------------------------------------
# 3. SELECT ONE FLOOR PLAN
# ------------------------------------------------------------

sample_global_index = int(train_indices[0])

# Copy only one small image from the memory-mapped cache.
sample_image = np.array(
    image_cache[sample_global_index],
    dtype=np.uint8,
    copy=True
)

print("\nSample information")
print("------------------")
print("Dataset index:", sample_global_index)
print("Record ID:", records[sample_global_index].get("id"))
print("Image shape:", sample_image.shape)
print("Image dtype:", sample_image.dtype)

assert sample_image.shape == (
    N_CHANNELS, IMAGE_SIZE, IMAGE_SIZE
)

# ------------------------------------------------------------
# 4. LOAD FONT
# ------------------------------------------------------------

try:
    font = ImageFont.truetype("arial.ttf", 13)
except OSError:
    font = ImageFont.load_default()

# ------------------------------------------------------------
# 5. CREATE 14-CHANNEL IMAGE GRID
# ------------------------------------------------------------

TILE_SIZE = 160
LABEL_HEIGHT = 30
GRID_COLUMNS = 7
GRID_ROWS = 2

canvas_width = GRID_COLUMNS * TILE_SIZE
canvas_height = GRID_ROWS * (TILE_SIZE + LABEL_HEIGHT)

channel_canvas = Image.new(
    "RGB",
    (canvas_width, canvas_height),
    "white"
)

draw = ImageDraw.Draw(channel_canvas)

for channel_index in range(N_CHANNELS):

    row = channel_index // GRID_COLUMNS
    col = channel_index % GRID_COLUMNS

    x = col * TILE_SIZE
    y = row * (TILE_SIZE + LABEL_HEIGHT)

    # Convert binary channel into grayscale image.
    channel = sample_image[channel_index] * 255

    channel_img = Image.fromarray(channel, mode="L")

    channel_img = channel_img.resize(
        (TILE_SIZE - 10, TILE_SIZE - 10),
        Image.Resampling.NEAREST
    )

    channel_canvas.paste(
        channel_img.convert("RGB"),
        (x + 5, y + 5)
    )

    label = GEOMETRY_CHANNELS[channel_index]

    draw.text(
        (x + 5, y + TILE_SIZE),
        label,
        fill="black",
        font=font
    )

# Save channel visualisation.
channel_path = FIGURES_DIR / "dataset_channels.png"

channel_canvas.save(
    channel_path,
    format="PNG",
    optimize=True
)

print("\nChannel image saved:", channel_path)

# ------------------------------------------------------------
# 6. CREATE COMPOSITE FLOOR PLAN
# ------------------------------------------------------------

composite = np.max(sample_image, axis=0)

composite_img = Image.fromarray(
    composite * 255,
    mode="L"
)

composite_img = composite_img.resize(
    (512, 512),
    Image.Resampling.NEAREST
)

composite_canvas = Image.new(
    "RGB",
    (512, 550),
    "white"
)

composite_canvas.paste(
    composite_img.convert("RGB"),
    (0, 0)
)

composite_draw = ImageDraw.Draw(composite_canvas)

record_id = records[sample_global_index].get("id")

composite_draw.text(
    (10, 520),
    f"ResPlan Floor Plan | Record ID: {record_id}",
    fill="black",
    font=font
)

composite_path = FIGURES_DIR / "dataset_composite.png"

composite_canvas.save(
    composite_path,
    format="PNG",
    optimize=True
)

print("Composite image saved:", composite_path)

# ------------------------------------------------------------
# 7. RELEASE TEMPORARY MEMORY
# ------------------------------------------------------------

del sample_image
del composite
del composite_img
del composite_canvas
del channel_canvas
del draw
del composite_draw

gc.collect()

memory_status("After visualisation")

print("\nVISUALISATION COMPLETED")


========== Before visualisation ==========
Available RAM: 3.85 GB
System RAM usage: 75.5%
Jupyter RAM: 1.43 GB

Sample information
------------------
Dataset index: 9173
Record ID: 3822
Image shape: (14, 64, 64)
Image dtype: uint8

Channel image saved: D:\02_LEARNING\02_Masters_IAAC_AI_Architecture\10-Term_01\01-Ai Methods\hackathon-team4-resplan\01_autoencoder\figures\dataset_channels.png
Composite image saved: D:\02_LEARNING\02_Masters_IAAC_AI_Architecture\10-Term_01\01-Ai Methods\hackathon-team4-resplan\01_autoencoder\figures\dataset_composite.png

========== After visualisation ==========
Available RAM: 3.85 GB
System RAM usage: 75.5%
Jupyter RAM: 1.43 GB

VISUALISATION COMPLETED


## 8. PyTorch Dataset and DataLoaders

The dataset reads images from the memory-mapped cache and converts each batch to float32 in the range [0, 1]. Numeric conditioning features are already standardised using the training-only scaler.

In [9]:
# ============================================================
# SECTION 8 — PYTORCH DATASET AND DATALOADERS
# ResPlan Conditional VAE
# ============================================================

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import gc

# ------------------------------------------------------------
# 1. VERIFY VARIABLES FROM PREVIOUS SECTIONS
# ------------------------------------------------------------

required_variables = [
    "train_indices",
    "val_indices",
    "test_indices",
    "image_cache",
    "numeric_scaled",
    "NUMERIC_FEATURES",
    "GEOMETRY_CHANNELS",
    "N_CHANNELS",
    "IMAGE_SIZE",
    "DEVICE"
]

missing = [
    variable for variable in required_variables
    if variable not in globals()
]

if missing:
    raise RuntimeError(
        f"Missing variables: {missing}. "
        "Execute Sections 1–7 in order."
    )

# Check numeric conditioning data.
assert numeric_scaled.shape == (
    len(records),
    len(NUMERIC_FEATURES)
)

# Check image cache.
assert image_cache.shape == (
    len(records),
    N_CHANNELS,
    IMAGE_SIZE,
    IMAGE_SIZE
)

print("Input variables verified.")
print("Image cache:", image_cache.shape)
print("Numeric conditions:", numeric_scaled.shape)


# ------------------------------------------------------------
# 2. PYTORCH DATASET CLASS
# ------------------------------------------------------------

class ResPlanDataset(Dataset):

    def __init__(self, indices, image_cache, numeric_scaled):

        self.indices = np.asarray(
            indices,
            dtype=np.int64
        )

        self.image_cache = image_cache
        self.numeric_scaled = numeric_scaled

    def __len__(self):

        return len(self.indices)

    def __getitem__(self, index):

        # Original dataset index
        global_index = int(self.indices[index])

        # ----------------------------------------
        # IMAGE INPUT
        # ----------------------------------------

        # Cache is uint8 with binary values 0 and 1.
        # Convert to float32 WITHOUT dividing by 255.

        image = np.array(
            self.image_cache[global_index],
            dtype=np.float32,
            copy=True
        )

        image_tensor = torch.from_numpy(image)

        # ----------------------------------------
        # NUMERIC CONDITIONING INPUT
        # ----------------------------------------

        # Already imputed and standardised in Section 5.

        condition = np.array(
            self.numeric_scaled[global_index],
            dtype=np.float32,
            copy=True
        )

        condition_tensor = torch.from_numpy(condition)

        return image_tensor, condition_tensor


# ------------------------------------------------------------
# 3. CREATE DATASETS
# ------------------------------------------------------------

train_dataset = ResPlanDataset(
    train_indices,
    image_cache,
    numeric_scaled
)

val_dataset = ResPlanDataset(
    val_indices,
    image_cache,
    numeric_scaled
)

test_dataset = ResPlanDataset(
    test_indices,
    image_cache,
    numeric_scaled
)

print("\nDatasets created.")

print("Training:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Testing:", len(test_dataset))


# ------------------------------------------------------------
# 4. MEMORY-SAFE DATALOADER CONFIGURATION
# ------------------------------------------------------------

BATCH_SIZE = 16
NUM_WORKERS = 0
PIN_MEMORY = False

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=False
)


# ------------------------------------------------------------
# 5. TEST ONE BATCH
# ------------------------------------------------------------

xb, cb = next(iter(train_loader))

print("\n========== DATALOADER VERIFICATION ==========")

print("Image batch shape:", tuple(xb.shape))
print("Condition batch shape:", tuple(cb.shape))

print("Image dtype:", xb.dtype)
print("Condition dtype:", cb.dtype)

print("Image minimum:", float(xb.min()))
print("Image maximum:", float(xb.max()))

print("Condition features:", NUMERIC_FEATURES)

print("Batch size:", BATCH_SIZE)
print("DataLoader workers:", NUM_WORKERS)
print("Pin memory:", PIN_MEMORY)


# ------------------------------------------------------------
# 6. VALIDATE INPUTS FOR CONDITIONAL VAE
# ------------------------------------------------------------

assert xb.shape == (
    BATCH_SIZE,
    N_CHANNELS,
    IMAGE_SIZE,
    IMAGE_SIZE
)

assert cb.shape == (
    BATCH_SIZE,
    len(NUMERIC_FEATURES)
)

assert xb.dtype == torch.float32
assert cb.dtype == torch.float32

assert torch.isfinite(xb).all()
assert torch.isfinite(cb).all()

assert float(xb.min()) >= 0.0
assert float(xb.max()) <= 1.0

print("\nAll DataLoader checks PASSED.")

# Release test batch iterator memory.
del xb, cb
gc.collect()

Input variables verified.
Image cache: (17000, 14, 64, 64)
Numeric conditions: (17000, 3)

Datasets created.
Training: 11900
Validation: 2550
Testing: 2550

========== DATALOADER VERIFICATION ==========
Image batch shape: (16, 14, 64, 64)
Condition batch shape: (16, 3)
Image dtype: torch.float32
Condition dtype: torch.float32
Image minimum: 0.0
Image maximum: 1.0
Condition features: ['net_area', 'area', 'wall_depth']
Batch size: 16
DataLoader workers: 0
Pin memory: False

All DataLoader checks PASSED.


0

## 9. Conditional Variational Autoencoder architecture

The encoder maps a floor-plan tensor and its numeric condition to a latent mean (`mu`) and log variance (`logvar`). The reparameterisation trick samples `z` while allowing gradients to flow through the network.

The decoder receives both `z` and the numeric condition and predicts one output logit map per geometry channel.

The output is interpreted as per-pixel probabilities after applying sigmoid.

In [10]:
import torch
import torch.nn as nn
import gc

# --------------------------------------------------
# 1. Conditional VAE Architecture
# --------------------------------------------------

class ConditionalVAE(nn.Module):

    def __init__(
        self,
        in_channels=N_CHANNELS,
        condition_dim=len(NUMERIC_FEATURES),
        latent_dim=64
    ):
        super().__init__()

        self.latent_dim = latent_dim

        # -------------------------
        # Encoder: 64x64 -> 4x4
        # -------------------------

        self.encoder_conv = nn.Sequential(

            nn.Conv2d(in_channels, 32, 4, 2, 1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(32, 64, 4, 2, 1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(64, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(128, 256, 4, 2, 1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True)
        )

        # -------------------------
        # Condition Encoder
        # -------------------------

        self.condition_encoder = nn.Sequential(
            nn.Linear(condition_dim, 32),
            nn.LeakyReLU(0.2, inplace=True)
        )

        # -------------------------
        # Latent Space
        # -------------------------

        encoder_features = 256 * 4 * 4

        self.fc_mu = nn.Linear(
            encoder_features + 32,
            latent_dim
        )

        self.fc_logvar = nn.Linear(
            encoder_features + 32,
            latent_dim
        )

        # -------------------------
        # Decoder Input
        # -------------------------

        self.decoder_input = nn.Linear(
            latent_dim + 32,
            encoder_features
        )

        # -------------------------
        # Decoder: 4x4 -> 64x64
        # -------------------------

        self.decoder_conv = nn.Sequential(

            nn.ConvTranspose2d(256, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(128, 64, 4, 2, 1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(64, 32, 4, 2, 1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(32, in_channels, 4, 2, 1)
        )

    # -------------------------
    # Encoder Forward
    # -------------------------

    def encode(self, x, condition):

        features = self.encoder_conv(x)
        features = features.flatten(start_dim=1)

        condition_features = self.condition_encoder(condition)

        combined = torch.cat(
            [features, condition_features],
            dim=1
        )

        mu = self.fc_mu(combined)
        logvar = self.fc_logvar(combined)

        return mu, logvar

    # -------------------------
    # Reparameterization
    # -------------------------

    def reparameterize(self, mu, logvar):

        std = torch.exp(0.5 * logvar)
        epsilon = torch.randn_like(std)

        return mu + epsilon * std

    # -------------------------
    # Decoder Forward
    # -------------------------

    def decode(self, z, condition):

        condition_features = self.condition_encoder(condition)

        combined = torch.cat(
            [z, condition_features],
            dim=1
        )

        features = self.decoder_input(combined)

        features = features.view(-1, 256, 4, 4)

        output = self.decoder_conv(features)

        return output

    # -------------------------
    # Complete VAE Forward
    # -------------------------

    def forward(self, x, condition):

        mu, logvar = self.encode(x, condition)

        z = self.reparameterize(mu, logvar)

        logits = self.decode(z, condition)

        return logits, mu, logvar


# --------------------------------------------------
# 2. Retrieve One Training Batch
# --------------------------------------------------

xb, cb = next(iter(train_loader))

print("Batch image shape:", tuple(xb.shape))
print("Batch condition shape:", tuple(cb.shape))


# --------------------------------------------------
# 3. Model Shape Test
# --------------------------------------------------

shape_model = ConditionalVAE(latent_dim=64)

shape_model.eval()

with torch.no_grad():

    test_logits, test_mu, test_logvar = shape_model(
        xb[:2],
        cb[:2]
    )


# --------------------------------------------------
# 4. Display Output Shapes
# --------------------------------------------------

print("\nModel Shape Test Results")

print("Input:", tuple(xb[:2].shape))
print("Output logits:", tuple(test_logits.shape))
print("Latent mean:", tuple(test_mu.shape))
print("Latent log variance:", tuple(test_logvar.shape))


# --------------------------------------------------
# 5. Memory Cleanup
# --------------------------------------------------

del shape_model
del test_logits, test_mu, test_logvar, xb, cb

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\nShape test completed successfully.")

Batch image shape: (16, 14, 64, 64)
Batch condition shape: (16, 3)

Model Shape Test Results
Input: (2, 14, 64, 64)
Output logits: (2, 14, 64, 64)
Latent mean: (2, 64)
Latent log variance: (2, 64)

Shape test completed successfully.


## 10. VAE loss function

The objective combines:

1. **Reconstruction loss:** binary cross-entropy between the input geometry layers and reconstructed pixel probabilities.
2. **KL divergence:** regularises the learned latent distribution towards a standard normal distribution.

`BETA` controls the strength of the KL regularisation. We keep it fixed across all three model setups for a fair comparison.

In [11]:
# Keep this value identical for training, validation and test evaluation.
# The previously trained checkpoints used beta=1.0 with summed BCE and KL
# divided by batch size (equivalent to per-sample objective).
BETA = 1.0

def vae_loss_function(logits, targets, mu, logvar, beta=BETA):
    batch_size = targets.shape[0]

    reconstruction = F.binary_cross_entropy_with_logits(
        logits, targets, reduction="sum"
    ) / batch_size

    kl = -0.5 * torch.sum(
        1 + logvar - mu.pow(2) - logvar.exp()
    ) / batch_size

    total = reconstruction + beta * kl
    return total, reconstruction, kl


## 11. Training and evaluation functions

All three configurations use the same split, seed, batch size, Adam optimiser, learning rate, beta-weighted loss and 30-epoch limit. The best checkpoint is selected using validation objective. Checkpoints and histories are saved during training so progress survives a kernel restart.


In [12]:
LEARNING_RATE = 1e-3

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)

    total_loss = 0.0
    total_reconstruction = 0.0
    total_kl = 0.0
    total_samples = 0

    for images, conditions in loader:
        images = images.to(DEVICE, dtype=torch.float32)
        conditions = conditions.to(DEVICE, dtype=torch.float32)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            if training:
                logits, mu, logvar = model(images, conditions)
            else:
                # Deterministic validation/test: decode posterior mean.
                mu, logvar = model.encode(images, conditions)
                logits = model.decode(mu, conditions)

            loss, reconstruction, kl = vae_loss_function(
                logits, images, mu, logvar
            )

            if training:
                loss.backward()
                optimizer.step()

        batch_size = images.size(0)
        total_loss += loss.detach().item()
        total_reconstruction += reconstruction.detach().item()
        total_kl += kl.detach().item()
        total_samples += batch_size

        del images, conditions, logits, mu, logvar
        del loss, reconstruction, kl

    if total_samples == 0:
        raise RuntimeError("DataLoader produced no samples.")

    return {
        "loss": total_loss / total_samples,
        "reconstruction": total_reconstruction / total_samples,
        "kl": total_kl / total_samples
    }

def train_one_model(latent_dim, model_name):
    # Preserve the original experiment setup: Adam, fixed learning rate,
    # beta=1.0, 30 epochs, same split and same batch size for all models.
    model = ConditionalVAE(latent_dim=latent_dim).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    best_val_loss = float("inf")
    best_state = None
    history = []
    start_time = time.time()

    checkpoint_path = MODEL_DIR / f"{model_name}_best.pt"
    history_path = CACHE_DIR / f"{model_name}_history.csv"
    metadata_path = CACHE_DIR / f"{model_name}_metadata.json"

    for epoch in range(1, EPOCHS + 1):
        train_metrics = run_epoch(model, train_loader, optimizer)
        val_metrics = run_epoch(model, val_loader)

        row = {
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_reconstruction": train_metrics["reconstruction"],
            "train_kl": train_metrics["kl"],
            "val_loss": val_metrics["loss"],
            "val_reconstruction": val_metrics["reconstruction"],
            "val_kl": val_metrics["kl"]
        }
        history.append(row)

        if val_metrics["loss"] < best_val_loss:
            best_val_loss = val_metrics["loss"]
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }

            # Save the best checkpoint immediately.
            torch.save(
                {
                    "epoch": epoch,
                    "latent_dim": latent_dim,
                    "model_state_dict": best_state,
                    "best_val_loss": best_val_loss,
                    "beta": BETA,
                    "seed": SEED
                },
                checkpoint_path
            )

        # Save history after every epoch, so an interrupted run is recoverable.
        pd.DataFrame(history).to_csv(history_path, index=False)

        print(
            f"{model_name} | Epoch {epoch:02d}/{EPOCHS} | "
            f"train={train_metrics['loss']:.3f} | "
            f"val={val_metrics['loss']:.3f} | "
            f"best_val={best_val_loss:.3f}"
        )

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    elapsed_seconds = time.time() - start_time

    if best_state is None:
        raise RuntimeError(f"{model_name} did not produce a checkpoint.")

    metadata = {
        "model_name": model_name,
        "latent_dim": latent_dim,
        "training_seconds": elapsed_seconds,
        "best_validation_loss": best_val_loss,
        "epochs_completed": len(history),
        "beta": BETA,
        "seed": SEED,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "optimizer": "Adam"
    }
    metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

    del model, optimizer, best_state
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return pd.DataFrame(history), elapsed_seconds


## 12. Train three VAE configurations

We compare latent dimensions 32, 64 and 128. These are three setups of the same CVAE architecture; they differ only in latent capacity. Training runs sequentially and releases each model before starting the next configuration.


In [13]:
# Train the three latent-dimension configurations on the same split.
MODEL_CONFIGS = [
    {"name": "CVAE_z32", "latent_dim": 32},
    {"name": "CVAE_z64", "latent_dim": 64},
    {"name": "CVAE_z128", "latent_dim": 128}
]

training_histories = {}
training_times = {}
parameter_counts = {}

for config in MODEL_CONFIGS:
    name = config["name"]
    latent_dim = config["latent_dim"]

    print("\\n" + "=" * 65)
    print(f"Training {name} (latent_dim={latent_dim})")
    print("=" * 65)

    history, elapsed_seconds = train_one_model(latent_dim, name)

    training_histories[name] = history
    training_times[name] = elapsed_seconds

    # Parameter count is architecture-dependent; count on CPU.
    model_for_count = ConditionalVAE(latent_dim=latent_dim)
    parameter_counts[name] = count_parameters(model_for_count)
    del model_for_count
    gc.collect()

    print(f"Parameters: {parameter_counts[name]:,}")
    print(f"Training time: {elapsed_seconds / 60:.2f} minutes")

print("\\nAll three configurations completed.")


\n=================================================================
Training CVAE_z32 (latent_dim=32)
CVAE_z32 | Epoch 01/30 | train=463.354 | val=303.686 | best_val=303.686
CVAE_z32 | Epoch 02/30 | train=284.521 | val=265.086 | best_val=265.086
CVAE_z32 | Epoch 03/30 | train=266.964 | val=253.605 | best_val=253.605
CVAE_z32 | Epoch 04/30 | train=246.255 | val=244.175 | best_val=244.175
CVAE_z32 | Epoch 05/30 | train=235.449 | val=236.244 | best_val=236.244
CVAE_z32 | Epoch 06/30 | train=228.041 | val=230.974 | best_val=230.974
CVAE_z32 | Epoch 07/30 | train=221.964 | val=229.751 | best_val=229.751
CVAE_z32 | Epoch 08/30 | train=216.952 | val=225.951 | best_val=225.951
CVAE_z32 | Epoch 09/30 | train=212.524 | val=224.651 | best_val=224.651
CVAE_z32 | Epoch 10/30 | train=208.700 | val=221.052 | best_val=221.052
CVAE_z32 | Epoch 11/30 | train=204.805 | val=221.017 | best_val=221.017
CVAE_z32 | Epoch 12/30 | train=202.022 | val=234.519 | best_val=221.017
CVAE_z32 | Epoch 13/30 | train=201

In [13]:
for name in ["CVAE_z32", "CVAE_z64", "CVAE_z128"]:

    history = CACHE_DIR / f"{name}_history.csv"
    checkpoint = MODEL_DIR / f"{name}_best.pt"

    print(f"\n{name}")
    print("History exists:", history.exists())
    print("Checkpoint exists:", checkpoint.exists())


CVAE_z32
History exists: True
Checkpoint exists: True

CVAE_z64
History exists: True
Checkpoint exists: True

CVAE_z128
History exists: True
Checkpoint exists: True


## 13. Training curves

The figure compares training and validation objective values for the three latent sizes. The validation curve is important because it indicates whether reconstruction performance generalises beyond the training records.

In [14]:
import pandas as pd
import numpy as np
from PIL import Image, ImageDraw
from pathlib import Path

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

model_names = [
    "CVAE_z32",
    "CVAE_z64",
    "CVAE_z128"
]

training_histories = {}

for name in model_names:

    # Load saved training history
    history_path = CACHE_DIR / f"{name}_history.csv"

    if not history_path.exists():
        print("Missing history:", history_path)
        continue

    history = pd.read_csv(history_path)
    training_histories[name] = history

    # Extract values
    epochs = history["epoch"].to_numpy()
    train_loss = history["train_loss"].to_numpy()
    val_loss = history["val_loss"].to_numpy()

    # Image dimensions
    width, height = 900, 500
    left, right = 80, 40
    top, bottom = 50, 70

    image = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(image)

    # Plot boundaries
    x_min, x_max = epochs.min(), epochs.max()
    y_min = min(train_loss.min(), val_loss.min())
    y_max = max(train_loss.max(), val_loss.max())

    if y_max == y_min:
        y_max += 1

    # Add padding
    padding = (y_max - y_min) * 0.08
    y_min -= padding
    y_max += padding

    def map_x(value):
        return left + (value - x_min) / max(x_max - x_min, 1) * (width-left-right)

    def map_y(value):
        return height-bottom - (value-y_min)/(y_max-y_min) * (height-top-bottom)

    # Draw axes
    draw.line(
        [(left, top), (left, height-bottom), (width-right, height-bottom)],
        fill="black",
        width=2
    )

    # Draw grid
    for i in range(6):
        y = top + i * (height-top-bottom) / 5
        draw.line([(left, y), (width-right, y)], fill="#dddddd", width=1)

    # Draw training and validation curves
    train_points = [
        (map_x(x), map_y(y))
        for x, y in zip(epochs, train_loss)
    ]

    val_points = [
        (map_x(x), map_y(y))
        for x, y in zip(epochs, val_loss)
    ]

    if len(train_points) > 1:
        draw.line(train_points, fill="blue", width=3)

    if len(val_points) > 1:
        draw.line(val_points, fill="red", width=3)

    # Labels
    draw.text((left, 15), f"{name} Training History", fill="black")
    draw.text((left, height-35), "Epoch", fill="black")
    draw.text((left, height-55), "Blue: Training   Red: Validation", fill="black")

    # Save figure
    output_path = FIGURES_DIR / f"{name}_training_history.png"
    image.save(output_path)

    print(f"{name}: {len(history)} epochs loaded")
    print("Saved:", output_path)

print("\nTraining curves completed.")

CVAE_z32: 30 epochs loaded
Saved: D:\02_LEARNING\02_Masters_IAAC_AI_Architecture\10-Term_01\01-Ai Methods\hackathon-team4-resplan\01_autoencoder\figures\CVAE_z32_training_history.png
CVAE_z64: 30 epochs loaded
Saved: D:\02_LEARNING\02_Masters_IAAC_AI_Architecture\10-Term_01\01-Ai Methods\hackathon-team4-resplan\01_autoencoder\figures\CVAE_z64_training_history.png
CVAE_z128: 30 epochs loaded
Saved: D:\02_LEARNING\02_Masters_IAAC_AI_Architecture\10-Term_01\01-Ai Methods\hackathon-team4-resplan\01_autoencoder\figures\CVAE_z128_training_history.png

Training curves completed.


## 14. Evaluate all models on the held-out test set

After model training, the held-out test set is evaluated once for each saved checkpoint. Evaluation uses deterministic posterior-mean reconstruction and the same beta-weighted objective as training. CPU evaluation is used to reduce GPU memory pressure.


In [14]:
# Restore model configurations

MODEL_CONFIGS = [
    {"name": "CVAE_z32", "latent_dim": 32},
    {"name": "CVAE_z64", "latent_dim": 64},
    {"name": "CVAE_z128", "latent_dim": 128}
]

print("Model configurations restored:")
print(MODEL_CONFIGS)

Model configurations restored:
[{'name': 'CVAE_z32', 'latent_dim': 32}, {'name': 'CVAE_z64', 'latent_dim': 64}, {'name': 'CVAE_z128', 'latent_dim': 128}]


# 15 Save final model selection

In [15]:
# Select the model with the lowest best validation objective.
validation_scores = {}

for config in MODEL_CONFIGS:
    name = config["name"]
    history_path = CACHE_DIR / f"{name}_history.csv"

    if not history_path.exists():
        raise FileNotFoundError(f"Missing history: {history_path}")

    history = pd.read_csv(history_path)
    validation_scores[name] = float(history["val_loss"].min())

best_model_name = min(validation_scores, key=validation_scores.get)
latent_lookup = {item["name"]: item["latent_dim"] for item in MODEL_CONFIGS}
BEST_LATENT_DIM = latent_lookup[best_model_name]

print("Validation objectives:", validation_scores)
print("Selected model:", best_model_name)
print("Selected latent dimension:", BEST_LATENT_DIM)


Validation objectives: {'CVAE_z32': 214.25433383118872, 'CVAE_z64': 187.98840475643385, 'CVAE_z128': 191.90058900122548}
Selected model: CVAE_z64
Selected latent dimension: 64


In [16]:
import json
import gc

test_loader_cpu = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

test_results = []

for config in MODEL_CONFIGS:
    name = config["name"]
    latent_dim = config["latent_dim"]
    checkpoint_path = MODEL_DIR / f"{name}_best.pt"
    metadata_path = CACHE_DIR / f"{name}_metadata.json"

    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Missing checkpoint: {checkpoint_path}")

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=True
    )

    model = ConditionalVAE(latent_dim=latent_dim)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    total_loss = 0.0
    total_reconstruction = 0.0
    total_kl = 0.0
    total_samples = 0

    with torch.inference_mode():
        for images, conditions in test_loader_cpu:
            mu, logvar = model.encode(images, conditions)
            logits = model.decode(mu, conditions)

            loss, reconstruction, kl = vae_loss_function(
                logits, images, mu, logvar
            )

            batch_size = images.shape[0]
            total_loss += loss.item() * batch_size
            total_reconstruction += reconstruction.item() * batch_size
            total_kl += kl.item() * batch_size
            total_samples += batch_size

    metadata = (
        json.loads(metadata_path.read_text(encoding="utf-8"))
        if metadata_path.exists() else {}
    )

    test_results.append({
        "Model": name,
        "Latent Dimension": latent_dim,
        "Parameters": count_parameters(model),
        "Training Time (min)": round(
            metadata.get("training_seconds", 0.0) / 60, 2
        ),
        "Best Validation Objective": float(checkpoint["best_val_loss"]),
        "Test VAE Objective": total_loss / total_samples,
        "Test Reconstruction Loss": total_reconstruction / total_samples,
        "Test KL Divergence": total_kl / total_samples
    })

    print(
        f"{name}: test objective={total_loss / total_samples:.3f}, "
        f"reconstruction={total_reconstruction / total_samples:.3f}, "
        f"KL={total_kl / total_samples:.3f}"
    )

    del model, checkpoint
    gc.collect()

comparison_df = pd.DataFrame(test_results).sort_values(
    "Best Validation Objective"
).reset_index(drop=True)

comparison_df.to_csv(
    CACHE_DIR / "model_comparison.csv",
    index=False
)

display(comparison_df.round(4))
print("Test comparison saved.")


CVAE_z32: test objective=3416.840, reconstruction=3300.385, KL=116.455
CVAE_z64: test objective=3018.958, reconstruction=2755.395, KL=263.563
CVAE_z128: test objective=3074.671, reconstruction=2721.702, KL=352.969


,Model,Latent Dimension,Parameters,Training Time (min),Best Validation Objective,Test VAE Objective,Test Reconstruction Loss,Test KL Divergence
0,CVAE_z64,64,2318670,3.87,187.9884,3018.9584,2755.3952,263.5632
1,CVAE_z128,128,3109326,3.99,191.9006,3074.6712,2721.7019,352.9692
2,CVAE_z32,32,1923342,3.74,214.2543,3416.8398,3300.3850,116.4548


Test comparison saved.


The model comparison below reports the held-out test results. The selected model is determined using the lowest validation objective, not by choosing the lowest test score. Interpret differences in reconstruction and KL terms together; a larger latent dimension does not automatically produce better floor-plan reconstructions.


# Load the best model

In [17]:
BEST_MODEL_NAME = best_model_name
BEST_CHECKPOINT_PATH = MODEL_DIR / f"{BEST_MODEL_NAME}_best.pt"

checkpoint = torch.load(
    BEST_CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=True
)

best_model = ConditionalVAE(latent_dim=BEST_LATENT_DIM)
best_model.load_state_dict(checkpoint["model_state_dict"])
best_model.eval()

print("Selected model loaded on CPU.")
print("Model:", BEST_MODEL_NAME)
print("Latent dimension:", BEST_LATENT_DIM)
print("Parameters:", f"{count_parameters(best_model):,}")
print("Best validation objective:", checkpoint["best_val_loss"])


Selected model loaded on CPU.
Model: CVAE_z64
Latent dimension: 64
Parameters: 2,318,670
Best validation objective: 187.98840475643382


# 16 — Floor-plan Reconstruction

In [18]:
# One-sample deterministic reconstruction on CPU.
# This intentionally avoids GPU inference and Matplotlib rendering.
from PIL import Image, ImageDraw, ImageFont
import numpy as np
import gc

sample_image, sample_condition = test_dataset[0]
original = sample_image.unsqueeze(0).float()
condition = sample_condition.unsqueeze(0).float()

best_model.eval()
with torch.inference_mode():
    mu, logvar = best_model.encode(original, condition)
    reconstruction_logits = best_model.decode(mu, condition)
    reconstructed = torch.sigmoid(reconstruction_logits)

original_np = original[0].numpy()
reconstructed_np = reconstructed[0].numpy()

reconstruction_mae = float(
    np.mean(np.abs(original_np - reconstructed_np))
)
print(f"Mean absolute reconstruction error: {reconstruction_mae:.5f}")

channel_indices = list(range(7))
channel_names = [
    "Wall", "Door", "Window", "Front door",
    "Bedroom", "Bathroom", "Kitchen"
]

scale = 3
cell_size = IMAGE_SIZE * scale
label_height = 26
margin = 10
columns = len(channel_indices)

canvas_width = columns * cell_size + (columns + 1) * margin
canvas_height = 2 * (cell_size + label_height) + 3 * margin
canvas = Image.new("RGB", (canvas_width, canvas_height), "white")
draw = ImageDraw.Draw(canvas)

try:
    font = ImageFont.truetype("arial.ttf", 14)
except OSError:
    font = ImageFont.load_default()

for col, channel in enumerate(channel_indices):
    x = margin + col * (cell_size + margin)

    for row, data in enumerate((original_np, reconstructed_np)):
        y = margin + row * (cell_size + label_height + margin)

        pixels = (
            np.clip(data[channel], 0.0, 1.0) * 255
        ).astype(np.uint8)

        tile = Image.fromarray(pixels).resize(
            (cell_size, cell_size),
            Image.Resampling.NEAREST
        )
        canvas.paste(tile.convert("RGB"), (x, y + label_height))

        if row == 0:
            draw.text((x, y), channel_names[col], fill="black", font=font)

        if col == 0:
            row_label = "Original" if row == 0 else "Reconstructed"
            draw.text(
                (margin, y + label_height + cell_size + 2),
                row_label,
                fill="black",
                font=font
            )

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
reconstruction_path = FIGURES_DIR / f"{BEST_MODEL_NAME.lower()}_reconstruction.png"
canvas.save(reconstruction_path, format="PNG", optimize=True)

print("Reconstruction figure saved:", reconstruction_path)

# Keep original/reconstructed tensors available for the next optional analysis.
del sample_image, sample_condition, condition, mu, logvar, reconstruction_logits
gc.collect()


Mean absolute reconstruction error: 0.02644
Reconstruction figure saved: D:\02_LEARNING\02_Masters_IAAC_AI_Architecture\10-Term_01\01-Ai Methods\hackathon-team4-resplan\01_autoencoder\figures\cvae_z64_reconstruction.png


8

## 17. VAE latent-space blend

Two held-out floor plans are encoded to their posterior means. The latent vectors and numeric conditions are interpolated from endpoint A to endpoint B. The decoder generates the intermediate semantic-layer predictions. This is a qualitative demonstration, not a guarantee of valid architecture.


In [19]:
# VAE blend: interpolate between two held-out floor plans in latent space.
# The numeric condition is interpolated at the same time.
from PIL import Image, ImageDraw, ImageFont
import numpy as np

if len(test_dataset) < 2:
    raise RuntimeError("At least two test samples are required for interpolation.")

sample_a_image, sample_a_condition = test_dataset[0]
sample_b_image, sample_b_condition = test_dataset[1]

x_a = sample_a_image.unsqueeze(0).float()
c_a = sample_a_condition.unsqueeze(0).float()
x_b = sample_b_image.unsqueeze(0).float()
c_b = sample_b_condition.unsqueeze(0).float()

best_model.eval()
with torch.inference_mode():
    mu_a, _ = best_model.encode(x_a, c_a)
    mu_b, _ = best_model.encode(x_b, c_b)

    blend_outputs = []
    alphas = np.linspace(0.0, 1.0, 5)

    for alpha in alphas:
        alpha_tensor = torch.tensor([[float(alpha)]], dtype=torch.float32)
        z_blend = (1.0 - alpha_tensor) * mu_a + alpha_tensor * mu_b
        c_blend = (1.0 - alpha_tensor) * c_a + alpha_tensor * c_b
        probabilities = torch.sigmoid(best_model.decode(z_blend, c_blend))
        blend_outputs.append(probabilities[0].numpy())

# Show selected semantic channels across the interpolation.
blend_channels = [0, 1, 2, 3, 4, 5, 6]
blend_labels = ["Wall", "Door", "Window", "Front door", "Bedroom", "Bathroom", "Kitchen"]

tile = 128
label_h = 22
margin = 8
cols = len(alphas)
rows = len(blend_channels)

blend_canvas = Image.new(
    "RGB",
    (cols * (tile + margin) + margin, rows * (tile + label_h + margin) + margin),
    "white"
)
blend_draw = ImageDraw.Draw(blend_canvas)

try:
    blend_font = ImageFont.truetype("arial.ttf", 12)
except OSError:
    blend_font = ImageFont.load_default()

for col, alpha in enumerate(alphas):
    for row, channel in enumerate(blend_channels):
        x = margin + col * (tile + margin)
        y = margin + row * (tile + label_h + margin)

        # Threshold probabilities for a clear semantic-layer comparison.
        binary = (blend_outputs[col][channel] >= 0.5).astype(np.uint8) * 255
        tile_img = Image.fromarray(binary, mode="L").resize(
            (tile, tile), Image.Resampling.NEAREST
        )
        blend_canvas.paste(tile_img.convert("RGB"), (x, y + label_h))

        if row == 0:
            blend_draw.text(
                (x, y), f"Blend {alpha:.2f}", fill="black", font=blend_font
            )
        if col == 0:
            blend_draw.text(
                (x, y + label_h + tile + 1),
                blend_labels[row],
                fill="black",
                font=blend_font
            )

blend_path = FIGURES_DIR / f"{BEST_MODEL_NAME.lower()}_latent_blend.png"
blend_canvas.save(blend_path, format="PNG", optimize=True)

print("Latent interpolation figure saved:", blend_path)
print("Interpretation: alpha 0 is plan A; alpha 1 is plan B.")
print("Intermediate columns show decoder outputs along the latent interpolation.")

del sample_a_image, sample_a_condition, sample_b_image, sample_b_condition
del x_a, c_a, x_b, c_b, mu_a, mu_b, blend_outputs, blend_canvas, blend_draw
gc.collect()


Latent interpolation figure saved: D:\02_LEARNING\02_Masters_IAAC_AI_Architecture\10-Term_01\01-Ai Methods\hackathon-team4-resplan\01_autoencoder\figures\cvae_z64_latent_blend.png
Interpretation: alpha 0 is plan A; alpha 1 is plan B.
Intermediate columns show decoder outputs along the latent interpolation.


0

## 18. Export selected CVAE to ONNX

The exported ONNX model accepts:
- `floorplan`: float32 tensor with shape `[batch, 14, 64, 64]`, values in [0, 1].
- `condition`: float32 tensor with shape `[batch, 3]`, standardised using the saved training scaler.

The ONNX model returns reconstructed per-pixel probabilities with the same 14-channel shape.


In [ ]:
#%pip install onnx onnxruntime

Note: you may need to restart the kernel to use updated packages.


In [21]:
class CVAEExportWrapper(nn.Module):
    def __init__(self, vae):
        super().__init__()
        self.vae = vae

    def forward(self, floorplan, condition):
        mu, logvar = self.vae.encode(floorplan, condition)
        # Deterministic inference uses the latent mean.
        logits = self.vae.decode(mu, condition)
        return torch.sigmoid(logits)

MODEL_APP_DIR.mkdir(parents=True, exist_ok=True)
export_model = CVAEExportWrapper(best_model).eval().cpu()

example_images = torch.zeros(1, N_CHANNELS, IMAGE_SIZE, IMAGE_SIZE, dtype=torch.float32)
example_conditions = torch.zeros(1, len(NUMERIC_FEATURES), dtype=torch.float32)

onnx_path = MODEL_APP_DIR / "resplan_cvae.onnx"

torch.onnx.export(
    export_model,
    (example_images, example_conditions),
    str(onnx_path),
    input_names=["floorplan", "condition"],
    output_names=["reconstruction"],
    dynamic_axes={
        "floorplan": {0: "batch_size"},
        "condition": {0: "batch_size"},
        "reconstruction": {0: "batch_size"},
    },
    opset_version=17,
    do_constant_folding=True,
    dynamo=False
)

print("ONNX exported:", onnx_path)
print("ONNX size (MB):", round(onnx_path.stat().st_size / (1024**2), 2))

C:\Users\FawadSaleem\AppData\Local\Temp\ipykernel_9940\518623122.py:20: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


ONNX exported: D:\02_LEARNING\02_Masters_IAAC_AI_Architecture\10-Term_01\01-Ai Methods\hackathon-team4-resplan\01_autoencoder\model_app\resplan_cvae.onnx
ONNX size (MB): 7.84


## 19. Validate ONNX against PyTorch

This check compares deterministic reconstruction probabilities from PyTorch and ONNX Runtime using actual held-out examples. The outputs should match within a small numerical tolerance.


In [22]:
import onnx
import onnxruntime as ort
import numpy as np
import torch

onnx_model = onnx.load(str(onnx_path))
onnx.checker.check_model(onnx_model)

session = ort.InferenceSession(
    str(onnx_path),
    providers=["CPUExecutionProvider"]
)

# Build a small real test batch without changing the held-out data.
validation_samples = [test_dataset[i] for i in range(2)]
validation_images = torch.stack([item[0] for item in validation_samples]).numpy().astype(np.float32)
validation_conditions = torch.stack([item[1] for item in validation_samples]).numpy().astype(np.float32)

with torch.inference_mode():
    pytorch_output = export_model(
        torch.from_numpy(validation_images),
        torch.from_numpy(validation_conditions)
    ).numpy()

onnx_output = session.run(
    ["reconstruction"],
    {
        "floorplan": validation_images,
        "condition": validation_conditions
    }
)[0]

max_abs_difference = float(np.max(np.abs(pytorch_output - onnx_output)))
mean_abs_difference = float(np.mean(np.abs(pytorch_output - onnx_output)))

print("ONNX model checker: PASS")
print("Maximum absolute difference:", max_abs_difference)
print("Mean absolute difference:", mean_abs_difference)

assert np.allclose(pytorch_output, onnx_output, atol=1e-4, rtol=1e-3), (
    "ONNX output does not sufficiently match PyTorch."
)
print("PyTorch ↔ ONNX numerical validation: PASS")


ONNX model checker: PASS
Maximum absolute difference: 4.112720489501953e-06
Mean absolute difference: 4.532037323201621e-08
PyTorch ↔ ONNX numerical validation: PASS


## 20. Save example inputs for the model application

The example inputs use the same preprocessing and standardisation as training. They are stored as NumPy arrays for later API/server integration.


In [23]:
EXAMPLE_INPUT_DIR = MODEL_APP_DIR / "example_inputs"
EXAMPLE_INPUT_DIR.mkdir(parents=True, exist_ok=True)

for i in range(min(3, len(test_dataset))):
    image_tensor, condition_tensor = test_dataset[i]
    np.save(
        EXAMPLE_INPUT_DIR / f"floorplan_{i+1:02d}.npy",
        image_tensor.numpy().astype(np.float32)
    )
    np.save(
        EXAMPLE_INPUT_DIR / f"condition_{i+1:02d}.npy",
        condition_tensor.numpy().astype(np.float32)
    )

print("Saved example inputs to:", EXAMPLE_INPUT_DIR)

Saved example inputs to: D:\02_LEARNING\02_Masters_IAAC_AI_Architecture\10-Term_01\01-Ai Methods\hackathon-team4-resplan\01_autoencoder\model_app\example_inputs


In [24]:
# Save deployment configuration and preprocessing metadata.
model_config = {
    "model_name": BEST_MODEL_NAME,
    "architecture": "ConditionalVAE",
    "latent_dim": int(BEST_LATENT_DIM),
    "input_name": "floorplan",
    "input_shape": [None, N_CHANNELS, IMAGE_SIZE, IMAGE_SIZE],
    "condition_name": "condition",
    "condition_shape": [None, len(NUMERIC_FEATURES)],
    "output_name": "reconstruction",
    "output_shape": [None, N_CHANNELS, IMAGE_SIZE, IMAGE_SIZE],
    "output_activation": "sigmoid",
    "geometry_channels": GEOMETRY_CHANNELS,
    "numeric_features": NUMERIC_FEATURES,
    "numeric_imputation_medians": [float(x) for x in numeric_medians],
    "numeric_scaler_mean": [float(x) for x in numeric_scaler.mean_],
    "numeric_scaler_scale": [float(x) for x in numeric_scaler.scale_],
    "image_size": int(IMAGE_SIZE),
    "image_value_range": [0.0, 1.0],
    "beta": float(BETA),
    "seed": int(SEED),
    "inference": "deterministic posterior mean; decode(mu, condition)"
}

config_path = MODEL_APP_DIR / "config.json"
config_path.write_text(json.dumps(model_config, indent=2), encoding="utf-8")

print("Model configuration saved:", config_path)


Model configuration saved: D:\02_LEARNING\02_Masters_IAAC_AI_Architecture\10-Term_01\01-Ai Methods\hackathon-team4-resplan\01_autoencoder\model_app\config.json


## 21. Results and interpretation

Use the generated comparison table and figures to write evidence-based conclusions. Do not claim that a generated floor plan is architecturally valid solely because it looks plausible.

Discuss:
- Which latent dimension achieved the best validation and test objective?
- Did validation loss decrease with training?
- How well were individual semantic layers reconstructed?
- Does latent interpolation produce gradual, interpretable changes?
- Which features were poorly reconstructed?
- What are the limitations of raster output and the current conditioning variables?


In [25]:
display(comparison_df.round(4))

print("Selected model (validation):", BEST_MODEL_NAME)
print("Figures saved in:", FIGURES_DIR)
print("Deployment assets saved in:", MODEL_APP_DIR)

print("\\nGenerated figures:")
for path in sorted(FIGURES_DIR.glob("*.png")):
    print("-", path.name)

print("\\nModel application assets:")
for path in sorted(MODEL_APP_DIR.rglob("*")):
    if path.is_file():
        print("-", path.relative_to(MODEL_APP_DIR))


,Model,Latent Dimension,Parameters,Training Time (min),Best Validation Objective,Test VAE Objective,Test Reconstruction Loss,Test KL Divergence
0,CVAE_z64,64,2318670,3.87,187.9884,3018.9584,2755.3952,263.5632
1,CVAE_z128,128,3109326,3.99,191.9006,3074.6712,2721.7019,352.9692
2,CVAE_z32,32,1923342,3.74,214.2543,3416.8398,3300.3850,116.4548


Selected model (validation): CVAE_z64
Figures saved in: D:\02_LEARNING\02_Masters_IAAC_AI_Architecture\10-Term_01\01-Ai Methods\hackathon-team4-resplan\01_autoencoder\figures
Deployment assets saved in: D:\02_LEARNING\02_Masters_IAAC_AI_Architecture\10-Term_01\01-Ai Methods\hackathon-team4-resplan\01_autoencoder\model_app
\nGenerated figures:
- CVAE_z128_training_history.png
- CVAE_z32_training_history.png
- cvae_z64_latent_blend.png
- cvae_z64_reconstruction.png
- CVAE_z64_training_history.png
- dataset_channels.png
- dataset_composite.png
- reconstructions.png
- training_curve.png
\nModel application assets:
- .gitkeep
- config.json
- example_inputs\condition_01.npy
- example_inputs\condition_02.npy
- example_inputs\condition_03.npy
- example_inputs\floorplan_01.npy
- example_inputs\floorplan_02.npy
- example_inputs\floorplan_03.npy
- resplan_cvae.onnx


## 22. Limitations and next steps

1. **Raster representation:** the output is a multi-channel raster, not vector CAD geometry.
2. **Architectural validity:** room adjacency, circulation, dimensions, accessibility and building regulations are not explicitly enforced.
3. **Neighbour graph:** the `neighbor` tuple is not yet integrated into the model. A future graph-based component could encode spatial connectivity.
4. **Resolution:** 64×64 is a computational compromise. Small doors, thin walls and fine geometric details may be lost.
5. **Conditional features:** only `net_area`, `area` and `wall_depth` are used as numeric conditions. More design constraints can be added after validating their meaning and data quality.
6. **Generation quality:** latent interpolations require qualitative and quantitative review. The CVAE does not guarantee valid residential layouts.

